# LangChain Agents Tutorial

This notebook demonstrates how to create and use agents in LangChain. We'll build a ReAct agent that can search the web and get weather information.

## Step 1: Setting up the OpenAI API Key

First, we need to set up our OpenAI API key to use the language model.


In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

## Step 2: Installing Required Packages

We need to install the necessary LangChain packages and other dependencies for our agent.


In [ ]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0 langchain-community==0.4.1 langchain-core==1.0.4 requests==2.32.5 duckduckgo-search==8.1.1 ddgs==9.9.0 langsmith==0.4.42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.9/401.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.

In [ ]:
!pip show langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-core
Version: 1.0

## Step 3: Importing Core Libraries

Let's import the essential LangChain components and other libraries we'll need.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

## Step 4: Setting up the Search Tool

We'll use DuckDuckGo search as one of our tools to allow the agent to search for information on the web.


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

## Step 5: Creating a Custom Weather Tool

Here we define a custom tool that can fetch weather data for any city using the WeatherStack API. This tool will be available to our agent.


In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This tool fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=[Replace with API Key]]&query={city}' # Enter the weatherstack api key before use

  response = requests.get(url)

  return response.json()

## Step 6: Initializing the Language Model

We create an instance of the ChatOpenAI model that will power our agent's reasoning capabilities.


In [ ]:
# llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    # Enable tool calling for this model
    model_kwargs={"tool_choice": "auto"},
    api_key=GROQ_API_KEY
)

## Step 7: Importing Agent Components

We import the necessary components to create a ReAct agent and agent executor.


In [ ]:
from langchain.agents import create_agent
from langsmith import Client

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

## Step 8: Loading the Pre-defined Prompt

We pull a standard Reasoning prompt from LangChain Hub. This prompt template guides the agent on how to reason through problems and take actions.


In [ ]:
# Step 2: Pull the prompt from LangSmith Hub

client = Client()
prompt = client.pull_prompt("hwchase17/react") # pulls the standard ReAct agent prompt
prompt_template_string = prompt.template # Extract the template string from the PromptTemplate object

## Step 9: Creating the Agent

Now we create our agent by combining the language model, our tools (search and weather), and the ReAct prompt template.


In [ ]:
# Step 3: Create the agent manually with the pulled prompt
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt=prompt_template_string
)

## Step 10: Testing the Agent

Let's test our agent with a complex query that requires both searching for information and getting weather data. The agent will need to:
1. Find the capital of Madhya Pradesh
2. Get the current weather for that city


In [ ]:
# Step 5: Invoke
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Find the capital of Madhya Pradesh, then find it's current weather condition"}]}
)

print(response)

{'messages': [HumanMessage(content='What is the current market cap of NVIDIA', additional_kwargs={}, response_metadata={}, id='5d16b96b-04b9-4dd6-b737-c11e1134658a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer current market cap of NVIDIA. Need up-to-date info, use duckduckgo search.', 'tool_calls': [{'id': 'fc_922dc112-60d1-4222-a550-12ba2b1c48a9', 'function': {'arguments': '{"query":"NVIDIA current market cap"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 316, 'total_tokens': 373, 'completion_time': 0.118920514, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.004777531, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.048421598, 'total_time': 0.123698045}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e88ce9c728', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provide

In [ ]:
for messages in response['messages']:
  print(messages)

content='What is the current market cap of NVIDIA' additional_kwargs={} response_metadata={} id='5d16b96b-04b9-4dd6-b737-c11e1134658a'
content='' additional_kwargs={'reasoning_content': 'We need to answer current market cap of NVIDIA. Need up-to-date info, use duckduckgo search.', 'tool_calls': [{'id': 'fc_922dc112-60d1-4222-a550-12ba2b1c48a9', 'function': {'arguments': '{"query":"NVIDIA current market cap"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 316, 'total_tokens': 373, 'completion_time': 0.118920514, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.004777531, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.048421598, 'total_time': 0.123698045}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e88ce9c728', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--548c577c-51ba-42de-9

# Final Step: Deployment

We'll use gradio to deploy our AI Agents. To deploy we'll follow the following steps.

1. Add all our logic for AI Agent into a single function
2. Create an interface using gradio's classes
3. Launch the app

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [29]:
def get_response(query):
  response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
  )

  return response['messages'][-1].content

In [ ]:
iface = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        label="Ask a question to your AI Agent",
        placeholder="e.g., Find the capital of Madhya Pradesh, then find it's current weather condition",
        lines=2
    ),
    outputs=gr.Textbox(label="Response", lines=10),
    title="AI Agent with Web Access",
    description="This AI Agent has access to the internet. You can ask anything and it will search the web to get you your answer.",
    examples=[
        ["Differentiate between VectorDB and Vector Store"],
        ["What is RAG model?"],
        ["What is the current market cap of NVIDIA"]
    ]
)

In [31]:
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://074675226076a75638.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Summary

This notebook demonstrated how to:
- Set up LangChain agents with custom tools
- Use the ReAct framework for reasoning and acting
- Combine multiple tools (search and weather API) in a single agent
- Execute complex multi-step queries that require both information retrieval and data processing
- Deploy an AI Agent using gradio

The agent successfully found that Bhopal is the capital of Madhya Pradesh and retrieved its current weather conditions by using both the search tool and the weather API tool in sequence.
